# Data Cleaning

This notebook prepares the collected YouTube comments for exploratory analysis and sentiment analysis.

The raw dataset is kept unchanged. Cleaning is performed on a copy of the raw data, and the resulting dataset is saved to `data/processed/`.

The cleaning process focuses on:
- validating the collected data
- standardizing timestamps
- removing unnecessary whitespace
- handling URLs and HTML entities where appropriate
- preserving emojis and sentiment-bearing punctuation
- checking for duplicate or empty comments
- creating basic text features

In [3]:
import pandas as pd
import re
from html import unescape
from pathlib import Path

In [4]:
input_path = Path("../data/raw/youtube_comments_750.csv")

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (750, 8)


,domain,creator,video_id,video_title,comment_id,comment_text,like_count,published_at
0,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgyKp3aIQy7uBXdjrbl4AaABAg,比不過一大堆的三星chain stores,0,2026-09-27T15:01:13Z
1,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgyNnshRIqIucsxRtFt4AaABAg,Trx apple store一大堆人,0,2026-09-27T15:00:21Z
2,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgyzM3Eg57SS-6t7i394AaABAg,天天向上的沒有得到attention 反而向下的得到,0,2026-09-27T14:59:41Z
3,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgzZ-wrg8ogB9bPET6Z4AaABAg,someone should make a 1-time payment alternati...,0,2026-09-27T14:59:31Z
4,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,Ugzksq-vVqMBKf3Z9iV4AaABAg,一個公司天天向上另一個天天向下,0,2026-09-27T14:58:56Z


In [5]:
clean_df = df.copy()

clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   domain        750 non-null    str  
 1   creator       750 non-null    str  
 2   video_id      750 non-null    str  
 3   video_title   750 non-null    str  
 4   comment_id    750 non-null    str  
 5   comment_text  750 non-null    str  
 6   like_count    750 non-null    int64
 7   published_at  750 non-null    str  
dtypes: int64(1), str(7)
memory usage: 177.1 KB


In [6]:
print("Missing values:")
print(clean_df.isnull().sum())

print("\nDuplicate rows:", clean_df.duplicated().sum())
print("Duplicate comment IDs:", clean_df["comment_id"].duplicated().sum())

Missing values:
domain          0
creator         0
video_id        0
video_title     0
comment_id      0
comment_text    0
like_count      0
published_at    0
dtype: int64

Duplicate rows: 0
Duplicate comment IDs: 0


In [7]:
pd.set_option("display.max_colwidth", 200)

clean_df[
    ["domain", "comment_text"]
].sample(20, random_state=42)

,domain,comment_text
506,Lifestyle,You should put pictures up .. pictures of all the cats
357,Travel,You can fly a 🇱🇹 plss am live
133,Technology,"It’s amazing, when ai first came out my first thought was “this can probably help me with my poker study because it can set up scenarios and me and the computer can go through odds and ranges and ..."
250,Travel,Kiwi
299,Travel,Kiwi
680,Lifestyle,"this is the first video of yours i've seen and let me tell you... this really cheered me up, which i needed desperately. thanks. :')"
336,Travel,stefan🔥🐐
155,Technology,Arun needs a run from apple store to stop buying stuff
528,Lifestyle,I love the couch! That is perfect for a cat cafe!
736,Lifestyle,Bros House is Turning into a cat at this point


## Text Cleaning Strategy

The cleaning process is intentionally conservative because punctuation, emojis, capitalization, and short expressions can contain useful sentiment information.

The original `comment_text` column is preserved. A separate `clean_text` column is created for analysis.

The following transformations are applied:
- Decode HTML entities
- Remove URLs
- Normalize excessive whitespace
- Remove leading and trailing whitespace

Emojis, punctuation, and short comments are retained because they may contribute to sentiment classification.

In [8]:
def clean_comment(text):
    text = unescape(str(text))
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
clean_df["clean_text"] = clean_df["comment_text"].apply(clean_comment)

In [10]:
clean_df[
    ["comment_text", "clean_text"]
].sample(20, random_state=42)

,comment_text,clean_text
506,You should put pictures up .. pictures of all the cats,You should put pictures up .. pictures of all the cats
357,You can fly a 🇱🇹 plss am live,You can fly a 🇱🇹 plss am live
133,"It’s amazing, when ai first came out my first thought was “this can probably help me with my poker study because it can set up scenarios and me and the computer can go through odds and ranges and ...","It’s amazing, when ai first came out my first thought was “this can probably help me with my poker study because it can set up scenarios and me and the computer can go through odds and ranges and ..."
250,Kiwi,Kiwi
299,Kiwi,Kiwi
680,"this is the first video of yours i've seen and let me tell you... this really cheered me up, which i needed desperately. thanks. :')","this is the first video of yours i've seen and let me tell you... this really cheered me up, which i needed desperately. thanks. :')"
336,stefan🔥🐐,stefan🔥🐐
155,Arun needs a run from apple store to stop buying stuff,Arun needs a run from apple store to stop buying stuff
528,I love the couch! That is perfect for a cat cafe!,I love the couch! That is perfect for a cat cafe!
736,Bros House is Turning into a cat at this point,Bros House is Turning into a cat at this point


In [11]:
print(
    clean_df["comment_text"]
    .str.contains(r"\\n", regex=True, na=False)
    .sum()
)

0


In [12]:
clean_df[
    ["comment_text", "clean_text"]
].sample(20, random_state=42)

,comment_text,clean_text
506,You should put pictures up .. pictures of all the cats,You should put pictures up .. pictures of all the cats
357,You can fly a 🇱🇹 plss am live,You can fly a 🇱🇹 plss am live
133,"It’s amazing, when ai first came out my first thought was “this can probably help me with my poker study because it can set up scenarios and me and the computer can go through odds and ranges and ...","It’s amazing, when ai first came out my first thought was “this can probably help me with my poker study because it can set up scenarios and me and the computer can go through odds and ranges and ..."
250,Kiwi,Kiwi
299,Kiwi,Kiwi
680,"this is the first video of yours i've seen and let me tell you... this really cheered me up, which i needed desperately. thanks. :')","this is the first video of yours i've seen and let me tell you... this really cheered me up, which i needed desperately. thanks. :')"
336,stefan🔥🐐,stefan🔥🐐
155,Arun needs a run from apple store to stop buying stuff,Arun needs a run from apple store to stop buying stuff
528,I love the couch! That is perfect for a cat cafe!,I love the couch! That is perfect for a cat cafe!
736,Bros House is Turning into a cat at this point,Bros House is Turning into a cat at this point


In [13]:
empty_clean = clean_df["clean_text"].eq("").sum()

print("Empty cleaned comments:", empty_clean)

Empty cleaned comments: 0


In [14]:
changed = (clean_df["comment_text"] != clean_df["clean_text"]).sum()

print("Comments changed by cleaning:", changed)

Comments changed by cleaning: 57


In [15]:
clean_df["character_count"] = clean_df["clean_text"].str.len()

clean_df["word_count"] = clean_df["clean_text"].str.split().str.len()

In [16]:
clean_df[
    ["clean_text", "character_count", "word_count"]
].head(10)

,clean_text,character_count,word_count
0,比不過一大堆的三星chain stores,21,2
1,Trx apple store一大堆人,19,3
2,天天向上的沒有得到attention 反而向下的得到,26,2
3,"someone should make a 1-time payment alternative, like surely with local AI models it's not that difficult to analyze some data offline",135,22
4,一個公司天天向上另一個天天向下,15,1
5,必須知道哎,5,1
6,社會人群到底知不知道的,11,1
7,從來沒有過這樣子的日子咯,12,1
8,到底是什麼日子來的,9,1
9,什麼日子來的,6,1


In [17]:
clean_df[["character_count", "word_count"]].describe()

,character_count,word_count
count,750.000000,750.000000
mean,66.632000,12.794667
std,97.259679,17.908624
min,1.000000,1.000000
25%,20.000000,4.000000
50%,43.000000,8.000000
75%,78.000000,15.000000
max,1145.000000,195.000000


In [18]:
print("Rows:", len(clean_df))
print("Unique comment IDs:", clean_df["comment_id"].nunique())
print("Duplicate comment IDs:", clean_df["comment_id"].duplicated().sum())

Rows: 750
Unique comment IDs: 750
Duplicate comment IDs: 0


In [19]:
output_path = Path("../data/processed/youtube_comments_clean.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")

Saved cleaned dataset to: ..\data\processed\youtube_comments_clean.csv


In [20]:
processed_df = pd.read_csv(output_path)

print("Processed dataset shape:", processed_df.shape)
processed_df.head()

Processed dataset shape: (750, 11)


,domain,creator,video_id,video_title,comment_id,comment_text,like_count,published_at,clean_text,character_count,word_count
0,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgyKp3aIQy7uBXdjrbl4AaABAg,比不過一大堆的三星chain stores,0,2026-09-27T15:01:13Z,比不過一大堆的三星chain stores,21,2
1,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgyNnshRIqIucsxRtFt4AaABAg,Trx apple store一大堆人,0,2026-09-27T15:00:21Z,Trx apple store一大堆人,19,3
2,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgyzM3Eg57SS-6t7i394AaABAg,天天向上的沒有得到attention 反而向下的得到,0,2026-09-27T14:59:41Z,天天向上的沒有得到attention 反而向下的得到,26,2
3,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,UgzZ-wrg8ogB9bPET6Z4AaABAg,"someone should make a 1-time payment alternative, like surely with local AI models it's not that difficult to analyze some data offline",0,2026-09-27T14:59:31Z,"someone should make a 1-time payment alternative, like surely with local AI models it's not that difficult to analyze some data offline",135,22
4,Technology,Mrwhosetheboss,Oa3Ffr_04js,Eight Sleep just COOKED,Ugzksq-vVqMBKf3Z9iV4AaABAg,一個公司天天向上另一個天天向下,0,2026-09-27T14:58:56Z,一個公司天天向上另一個天天向下,15,1


## Cleaning Summary

The raw YouTube comment dataset contained 750 comments across 15 videos and three domains.

The cleaning process:
- preserved the original `comment_text`
- created a separate `clean_text` column
- decoded HTML entities
- removed URLs
- normalized whitespace
- preserved emojis and punctuation
- retained short comments
- verified that no comments became empty
- verified that all comment IDs remained unique
- added character and word count features

No rows were removed during the cleaning process.
The cleaned dataset was saved to `data/processed/youtube_comments_clean.csv`.